### Lid-driven cavity

This example shows that the pymrm toolbox is flexible enough to simulate
incompressible Navier-Stokes flow. The governing equations are the momentum
balance and the continuity (incompressibility) constraint:

$$
\rho \left( \frac{\partial \boldsymbol{v}}{\partial t} + \boldsymbol{v} \cdot \nabla \boldsymbol{v} \right) = -\nabla p + \mu \nabla^2 \boldsymbol{v}, \qquad \nabla \cdot \boldsymbol{v} = 0
$$

The setup is the classic lid-driven cavity: a closed square (or cubic) box
with no-slip walls. The top wall (the lid) moves tangentially at a constant
velocity `v_top`; no fluid passes through any wall. Because the box is closed,
the pressure has no boundary value of its own: the pressure correction gets a
zero normal gradient on every wall, and the pressure is fixed up to a constant
by pinning it in one cell. The flow is solved with a fractional-step (projection) method: a
momentum predictor gives a velocity field that is not yet divergence-free,
then a pressure-correction Poisson equation is solved and used to project the
velocity back onto the divergence-free subspace.

The setup is quite involved because of the staggered mesh: pressure lives at
cell centers, and each velocity component lives on its own staggered grid,
shifted by half a cell in its own direction. A surprising finding is that in
almost all cases the direct sparse solver is faster (or as fast) as ilu +
bicgstab.


### Imports

In [ ]:
import numpy as np
import scipy as sp
from scipy.sparse import linalg as sla
from scipy.sparse import eye_array
import matplotlib.pyplot as plt
from IPython.display import clear_output
from pymrm import interp_stagg_to_cntr, interp_cntr_to_stagg, interp_cntr_to_stagg_tvd
from pymrm import construct_grad, construct_div, minmod

### Physical and numerical parameters

In [ ]:
# Physical Parameters
dim = 2
rho = 1
mu = 1
p_ref = 0  # pressure in the reference cell (the pressure is fixed up to a constant)
v_top = 50
box = [1.0] * dim

# Numerical Parameters
shape_p = [50] * dim
Co = 0.5
dt = Co*(box[0]/v_top)/shape_p[0]
num_time_steps = 50

### Grid setup and boundary conditions

Pressure lives on a cell-centered grid; each velocity component `v[i]` lives
on a grid staggered by half a cell along its own axis `i` (a face-centered
grid in that direction, cell-centered in the others). Boundary conditions use
the OUTWARD normal, so the same `{'a', 'b', 'd'}` dictionary means opposite
gradients at the two ends of an axis. The lid velocity is imposed by setting
`bc_v[0][-1][1]['d'] = v_top`: the x-velocity (component 0), along the last
axis (`dim - 1`, pointing up), at its upper end (index 1). All other velocity
conditions are no-slip, and the normal velocity on every wall is zero by
construction: each velocity component is stored on the interior faces of its
own axis only, and the wall values are padded with zeros. The pressure gets a
zero normal gradient on all walls.

In [ ]:
x_cntr_c  = [None] * dim  # cell-centered positions
x_cntr_f  = [None] * dim  # cell face positions
x_stagg_c = [None] * dim  # staggered positions for velocities
x_stagg_f = [None] * dim  # face-positions corresponding to staggered cells
shape_v   = [None] * dim  # sizes velocity ndarrays
bc_v      = [None] * dim  # velocity bc
v         = [None] * dim
v_old     = [None] * dim
v_cntr    = [None] * dim
bc_p      = [None] * dim

for i in range(dim):
    shape_v[i] = shape_p.copy()
    x_cntr_f[i]  = np.linspace(0, box[i], shape_p[i]+1)
    x_cntr_c[i]  = 0.5*(x_cntr_f[i][:-1]+x_cntr_f[i][1:])
    x_stagg_c[i] = x_cntr_f[i][1:-1]
    x_stagg_f[i] = np.concatenate((x_cntr_f[i][0:1], x_cntr_c[i][1:-1], x_cntr_f[i][-1:]), axis=0)
    bc_p[i] = ({'a': 1, 'b': 0, 'd': 0},)*2
    bc_v[i] = [None] * dim
    for j in range(dim):
        shape_v[i][j] = shape_p[j]
        bc_v[i][j] = ({'a': 0, 'b': 1, 'd': 0}, {'a': 0, 'b': 1, 'd': 0})
    shape_v[i][i] -= 1
bc_v[0][-1][1]['d'] = v_top  # the lid moves in the x direction

### Operators: gradient, divergence, Laplacian, Jacobians, preconditioners

For pressure and for each velocity component, the gradient, divergence and
Laplacian operators are assembled once per axis and kept as constant sparse
matrices. The momentum Jacobian `jac_mat_v[i] = (rho/dt) * eye - mu *
lapl_mat_v[i]` is the implicit-Euler operator for the momentum predictor; an
ILU factorization of it, and of the pressure Laplacian, is built once and
reused as a preconditioner (and, for pressure, as a direct LU solve) inside
the time loop. With zero-gradient conditions on all walls the pressure
Laplacian is singular (a constant can be added to the pressure); replacing the
equation of one reference cell by `dp = 0` removes the constant. The other
equations still enforce continuity in that cell, because the divergences of a
closed box sum to zero.

In [ ]:
grad_mat_v    = [None] * dim
grad_bc_v = [None] * dim
div_mat_v     = [None] * dim
lapl_mat_v    = [None] * dim
lapl_bc_v = [None] * dim
jac_mat_v     = [None] * dim
jac_pc_v  = [None] * dim
g_v       = [None] * dim
grad_mat_p    = [None] * dim
grad_bc_p = [None] * dim
div_mat_p     = [None] * dim
p = p_ref*np.ones(shape_p)
lapl_mat_p    = sp.sparse.csc_array((p.size, p.size))
lapl_bc_p = sp.sparse.csc_array((p.size, 1))

for i in range(dim):
    grad_mat_p[i], grad_bc_p[i] = construct_grad(shape_p, x_cntr_f[i], x_cntr_c[i], bc_p[i], axis=i)
    div_mat_p[i] = construct_div(shape_p, x_cntr_f[i], nu=0, axis=i)
    lapl_mat_p = lapl_mat_p + div_mat_p[i] @ grad_mat_p[i]
    lapl_bc_p = lapl_bc_p + div_mat_p[i] @ grad_bc_p[i]

    grad_mat_v[i]    = [None] * dim
    grad_bc_v[i] = [None] * dim
    div_mat_v[i]     = [None] * dim
    v[i] = np.zeros(shape_v[i])
    lapl_mat_v[i] = sp.sparse.csc_array((v[i].size, v[i].size))
    lapl_bc_v[i] = sp.sparse.csc_array((v[i].size, 1))
    for j in range(dim):
        if (i == j):
            grad_mat_v[i][j], grad_bc_v[i][j] = construct_grad(shape_v[i], x_stagg_f[j], x_stagg_c[j], bc_v[i][j], axis=j)
            div_mat_v[i][j] = construct_div(shape_v[i], x_stagg_f[j], nu=0, axis=j)
        else:
            grad_mat_v[i][j], grad_bc_v[i][j] = construct_grad(shape_v[i], x_cntr_f[j], x_cntr_c[j], bc_v[i][j], axis=j)
            div_mat_v[i][j] = construct_div(shape_v[i], x_cntr_f[j], nu=0, axis=j)
        lapl_mat_v[i] = lapl_mat_v[i] + div_mat_v[i][j] @ grad_mat_v[i][j]
        lapl_bc_v[i] = lapl_bc_v[i] + div_mat_v[i][j] @ grad_bc_v[i][j]
    jac_mat_v[i] = (rho/dt) * eye_array(v[i].size, format='csc') - mu * lapl_mat_v[i]
    jac_ilu_v = sla.spilu(jac_mat_v[i])
    jac_pc_v[i] = sla.LinearOperator(jac_ilu_v.shape, lambda x: jac_ilu_v.solve(x))
# pin the pressure correction in the reference cell (index 0, bottom-left corner)
i_ref = 0
lapl_mat_p = lapl_mat_p.tolil()
lapl_mat_p[i_ref, :] = 0
lapl_mat_p[i_ref, i_ref] = 1
lapl_mat_p = lapl_mat_p.tocsc()
lapl_ilu_p = sla.spilu(lapl_mat_p)
lapl_lu_p = sla.splu(lapl_mat_p)
lapl_pc_p = sla.LinearOperator(lapl_ilu_p.shape, lambda x: lapl_ilu_p.solve(x))

### Time marching: projection method

Each time step performs a fractional-step (projection) update:

1. **Momentum predictor**: assemble the TVD-limited (minmod) convective flux
   and the pressure gradient from the previous step, then solve the implicit
   momentum equation for each velocity component with `bicgstab`,
   preconditioned by the ILU factorization built above.
2. **Pressure correction**: solve a Poisson equation for the pressure
   correction `dp` from the divergence of the predicted velocity, using the
   direct sparse LU solve `lapl_lu_p.solve(...)`.
3. **Velocity correction**: project the velocity back onto the
   divergence-free subspace using the pressure-correction gradient, and plot
   the pressure field and streamlines.


In [ ]:
X, Y = np.meshgrid(x_cntr_c[0], x_cntr_c[1])

for k in range(num_time_steps):
    div_v = np.zeros((p.size, 1))
    for i in range(dim):
        v_old[i] = v[i].copy()
        g_conv = np.zeros((v[i].size, 1))
        for j in range(dim):
            if (j == i):
                v_f = interp_cntr_to_stagg(v[i], x_stagg_f[i], x_stagg_c[i], axis=i)
                vi_f, dvi_f = interp_cntr_to_stagg_tvd(v[i], x_stagg_f[i], x_stagg_c[i], bc_v[i][j], v_f, minmod, axis=j)
                conv_flux = rho*vi_f*vi_f
            else:
                v_f = interp_stagg_to_cntr(v[j], x_cntr_c[i], x_stagg_c[i], axis=i)
                shape_v_f_bnd = list(v_f.shape)
                shape_v_f_bnd[j] = 1
                v_f = np.concatenate((np.zeros(shape_v_f_bnd), v_f, np.zeros(shape_v_f_bnd)), axis=j)
                vi_f, dvi_f = interp_cntr_to_stagg_tvd(v[i], x_cntr_f[j], x_cntr_c[j], bc_v[i][j], v_f, minmod, axis=j)
                conv_flux = rho*v_f*vi_f
            g_conv = g_conv + div_mat_v[i][j] @ conv_flux.reshape(-1, 1)
        shape_p_f = shape_p.copy()
        shape_p_f[i] = shape_p[i]+1
        grad_p = (grad_mat_p[i] @ p.reshape(-1, 1) + grad_bc_p[i]).reshape(shape_p_f)
        idx = [slice(None)] * dim
        idx[i] = slice(1, shape_p_f[i]-1)
        g_grad_p = grad_p[tuple(idx)].reshape(-1, 1)
        g_v[i] = jac_mat_v[i] @ v[i].reshape(-1, 1) - (rho/dt)*v_old[i].reshape(-1, 1) + g_conv + g_grad_p - mu*lapl_bc_v[i]
        dv, exit_code = sla.bicgstab(jac_mat_v[i], g_v[i], M=jac_pc_v[i])
        #jac_lu = sla.splu(jac_mat_v[i])
        #dv = jac_lu.solve(g_v[i])
        v[i] -= dv.reshape(shape_v[i])

        shape_v_bnd = list(v[i].shape)
        shape_v_bnd[i] = 1
        v_incl_wall = np.concatenate((np.zeros(shape_v_bnd), v[i], np.zeros(shape_v_bnd)), axis=i)
        v_cntr[i] = interp_stagg_to_cntr(v_incl_wall, x_cntr_f[i], x_cntr_c[i], axis=i)
        div_v += div_mat_p[i] @ v_incl_wall.reshape(-1, 1)

    #dp, exit_code = sla.bicgstab(lapl_mat_p, (rho/dt)*div_v, M=lapl_pc_p)
    rhs_p = (rho/dt)*div_v
    rhs_p[i_ref] = 0  # dp = 0 in the reference cell
    dp = lapl_lu_p.solve(rhs_p)
    p += dp.reshape(shape_p)
    for i in range(dim):
        shape_p_f = shape_p.copy()
        shape_p_f[i] = shape_p[i]+1
        grad_dp = (grad_mat_p[i] @ dp.reshape(-1, 1)).reshape(shape_p_f)
        idx = [slice(None)] * dim
        idx[i] = slice(1, shape_p_f[i]-1)
        v[i] -= (dt/rho) * grad_dp[tuple(idx)]
        shape_v_bnd = list(v[i].shape)
        shape_v_bnd[i] = 1
        v_incl_wall = np.concatenate((np.zeros(shape_v_bnd), v[i], np.zeros(shape_v_bnd)), axis=i)
        v_cntr[i] = interp_stagg_to_cntr(v_incl_wall, x_cntr_f[i], x_cntr_c[i], axis=i)
    clear_output(wait=True)
    plt.figure()
    contour = plt.pcolormesh(x_cntr_f[0], x_cntr_f[1], p.T, shading='flat', cmap='viridis')
    plt.streamplot(X, Y, v_cntr[0].T, v_cntr[1].T, color='white')
    plt.xticks([])
    plt.yticks([])
    plt.colorbar(contour, label='Pressure')
    plt.title(f"Time step: {k}")
    plt.show()

## Checking the result

The projection step exists to make the velocity field satisfy the discrete
continuity equation. Recompute the divergence of the final velocity in every
pressure cell, with the zero wall values padded exactly as in the time loop,
and compare it with the velocity scale `v_top / box[0]`. A relative value near
round-off confirms the projection; a large value in a few cells points at an
inconsistent boundary condition for the pressure correction (for example a
fixed pressure on a wall through which no fluid can flow).

In [ ]:
div_v_final = np.zeros((p.size, 1))
for i in range(dim):
    shape_v_bnd = list(v[i].shape)
    shape_v_bnd[i] = 1
    v_incl_wall = np.concatenate((np.zeros(shape_v_bnd), v[i], np.zeros(shape_v_bnd)), axis=i)
    div_v_final += div_mat_p[i] @ v_incl_wall.reshape(-1, 1)

div_scale = v_top / box[0]
print(f"max |div v| / (v_top / L) = {np.max(np.abs(div_v_final)) / div_scale:.2e}")
print(f"pressure in the reference cell: {p.ravel()[i_ref]:.3g} (p_ref = {p_ref})")